# 01- Intéraction simple avec un LLM

Dans ce premier exercice, nous allons poser les bases de notre agent d’IA en apprenant à interagir directement avec l'API d'un LLM provider.

L'objectif n’est pas encore de construire une logique complexe, mais simplement de comprendre comment envoyer une requête à un LLM et récupérer sa réponse en Python.

À la fin de cet exercice, vous serez capable d’appeler un modèle de langage depuis un notebook, de lui fournir un prompt textuel et d’afficher la réponse générée.

## Définition des variables
| Paramètre        | Description |
|------------------|-------------|
| Endpoint         | URL du service exposé par le fournisseur pour envoyer des requêtes au modèle. |
| Modèle           | Identifiant du modèle de langage à utiliser pour générer les réponses. |
| Clé d'API        | Permet de s’authentifier auprès du fournisseur de LLM et d’autoriser les appels à l’API. |
| Prompt / Messages| Texte ou structure de messages envoyés au modèle pour guider la génération. |


In [6]:
from dotenv import dotenv_values
config = dotenv_values("../../.env")

# Call to local lm studio
# API_URL = "http://localhost:1234/v1/chat/completions"
# API_KEY = ""
# MODEL_NAME = "qwen/qwen3-4b-2507"

# Call to remote llm
API_URL = "https://api.openai.com/v1/chat/completions"
API_KEY = config.get("LLM_API_KEY")
MODEL_NAME = "gpt-4.1-nano"

## Appel HTTP vers un LLM

Le code suivant effectue une requête HTTP POST vers le point d’accès du llm provider, en s’authentifiant à l’aide d’une clé d’API passée dans les en-têtes. Le corps de la requête précise le modèle utilisé, le message utilisateur qui sert de prompt.

Une fois la requête exécutée, la réponse est récupérée au format JSON.

In [7]:
import requests
from rich import print

response = requests.post(API_URL,
    headers={
        "Authorization": f"Bearer {API_KEY}"
    },
    json={
        "model": MODEL_NAME,
        "messages": [
            {
                "role": "user",
                "content": "List in few word the most popular development languages"
            }
        ],
        "frequency_penalty": 1.3 #  Les valeurs positives pénalisent les nouveaux tokens en fonction de leur fréquence actuelle dans le texte, ce qui réduit la probabilité que le modèle répète la même ligne mot pour mot.
    }
)

print(response.json())
print(response.json()['choices'][0]['message']['content'])

{
    'id': 'chatcmpl-Do8qSEvCaIgjZ93maXRoU0nvpKvfQ',
    'object': 'chat.completion',
    'created': 1780842772,
    'model': 'gpt-4.1-nano-2025-04-14',
    'choices': [
        {
            'index': 0,
            'message': {
                'role': 'assistant',
                'content': 'JavaScript, Python, Java, C++, C#, PHP.',
                'refusal': None,
                'annotations': []
            },
            'logprobs': None,
            'finish_reason': 'stop'
        }
    ],
    'usage': {
        'prompt_tokens': 16,
        'completion_tokens': 13,
        'total_tokens': 29,
        'prompt_tokens_details': {'cached_tokens': 0, 'audio_tokens': 0},
        'completion_tokens_details': {
            'reasoning_tokens': 0,
            'audio_tokens': 0,
            'accepted_prediction_tokens': 0,
            'rejected_prediction_tokens': 0
        }
    },
    'service_tier': 'default',
    'system_fingerprint': 'fp_96f2a865f6'
}

JavaScript, Python, Java, C++, C#, PHP.

## Pour aller plus loin: Demander le résumé d'un article
Le code suivant importe le contenu d'un article dans la variable `article_content`.

<ins>**Exercice:**</ins> Compléter ce code pour demander au llm de résumer l'article donc le contenu est mentionné

In [8]:
import sys
from pathlib import Path
import requests
from rich import print

sys.path.append(str(Path("../../utils").resolve()))
from file_reader import read_file

article_content = read_file(str(Path("../../assets/articles/001-article.md")))

response = requests.post(API_URL,
    headers={
        "Authorization": f"Bearer {API_KEY}"
    },
    json={
        "model": MODEL_NAME,
        "messages": [
            {
                "role": "user",
                "content": f"Résume l'article suivant: {article_content}"
            }
        ],
        "frequency_penalty": 1.3 #  Les valeurs positives pénalisent les nouveaux tokens en fonction de leur fréquence actuelle dans le texte, ce qui réduit la probabilité que le modèle répète la même ligne mot pour mot.
    }
)

print(response.json())
print(response.json()['choices'][0]['message']['content'])

{
    'id': 'chatcmpl-Do8qSPsCkLmaEUwiA5AtN5tPPxLe0',
    'object': 'chat.completion',
    'created': 1780842772,
    'model': 'gpt-4.1-nano-2025-04-14',
    'choices': [
        {
            'index': 0,
            'message': {
                'role': 'assistant',
                'content': 'La Révolution française (1789-1799) a profondément transformé la société, la politique 
et la culture françaises, avec des répercussions majeures en Europe. Elle a été déclenchée par une crise économique
grave, des inégalités sociales importantes entre le clergé, la noblesse et le tiers état (qui représentait 95 % de 
la population), ainsi que par l’influence des idées des Lumières prônant liberté, égalité et souveraineté du 
peuple. La convocation des États généraux en 1789 puis la prise de la Bastille ont marqué le début d’une série 
d’événements qui ont mené à l’abolition de la monarchie absolue, à l’instauration de droits fondamentaux comme ceux
inscrits dans la Déclaration des droits de l’homme et du citoyen, et à l’émergence d’une République.\n\nLe régime 
monarchique cédé progressivement sa place à une monarchie constitutionnelle puis à une République après l’exécution
de Louis XVI. La période dite de « Terreur », dirigée par Robespierre, fut marquée par les purges politiques et un 
climat d’insécurité extrême. Après sa chute en 1794, un gouvernement plus modéré appelé le Directoire prit le 
pouvoir mais resta instable jusqu’au coup d’État final réalisé par Napoléon Bonaparte en 1799. Ce dernier mit fin à
cette période révolutionnaire pour établir son contrôle.\n\nLes principales figures incluent Louis XVI ou 
Marie-Antoinette symbolisant la monarchie ; Robespierre et Danton pour leur rôle lors de cette révolution ; enfin 
Napoléon comme figure centrale qui complète cette époque turbulente tout en amorçant une nouvelle phase pour 
France.\n\nLes conséquences furent profondes : finibilitéelàdelaMonarchietouteprIrishRépublique , 
abolitionnedesprivilèges,humanismeet droitàl’individu.Mais aussi 
inspirationpourd’autresrévolutionsenEuropeetdanslemonde entier.Le texte souligne également quelques anecdotes clés 
telles que le temps exceptionnellement court pour rédiger certains textes fondateurs ou encore les effets 
économiques négatifs comme l’hyperinflation provoquée par une nouvelle monnaie révolutionnaire.\n',
                'refusal': None,
                'annotations': []
            },
            'logprobs': None,
            'finish_reason': 'stop'
        }
    ],
    'usage': {
        'prompt_tokens': 1439,
        'completion_tokens': 456,
        'total_tokens': 1895,
        'prompt_tokens_details': {'cached_tokens': 0, 'audio_tokens': 0},
        'completion_tokens_details': {
            'reasoning_tokens': 0,
            'audio_tokens': 0,
            'accepted_prediction_tokens': 0,
            'rejected_prediction_tokens': 0
        }
    },
    'service_tier': 'default',
    'system_fingerprint': 'fp_82a1439a62'
}

La Révolution française (1789-1799) a profondément transformé la société, la politique et la culture françaises, 
avec des répercussions majeures en Europe. Elle a été déclenchée par une crise économique grave, des inégalités 
sociales importantes entre le clergé, la noblesse et le tiers état (qui représentait 95 % de la population), ainsi 
que par l’influence des idées des Lumières prônant liberté, égalité et souveraineté du peuple. La convocation des 
États généraux en 1789 puis la prise de la Bastille ont marqué le début d’une série d’événements qui ont mené à 
l’abolition de la monarchie absolue, à l’instauration de droits fondamentaux comme ceux inscrits dans la 
Déclaration des droits de l’homme et du citoyen, et à l’émergence d’une République.

Le régime monarchique cédé progressivement sa place à une monarchie constitutionnelle puis à une République après 
l’exécution de Louis XVI. La période dite de « Terreur », dirigée par Robespierre, fut marquée par les purges 
politiques et un climat d’insécurité extrême. Après sa chute en 1794, un gouvernement plus modéré appelé le 
Directoire prit le pouvoir mais resta instable jusqu’au coup d’État final réalisé par Napoléon Bonaparte en 1799. 
Ce dernier mit fin à cette période révolutionnaire pour établir son contrôle.

Les principales figures incluent Louis XVI ou Marie-Antoinette symbolisant la monarchie ; Robespierre et Danton 
pour leur rôle lors de cette révolution ; enfin Napoléon comme figure centrale qui complète cette époque turbulente
tout en amorçant une nouvelle phase pour France.

Les conséquences furent profondes : finibilitéelàdelaMonarchietouteprIrishRépublique , 
abolitionnedesprivilèges,humanismeet droitàl’individu.Mais aussi 
inspirationpourd’autresrévolutionsenEuropeetdanslemonde entier.Le texte souligne également quelques anecdotes clés 
telles que le temps exceptionnellement court pour rédiger certains textes fondateurs ou encore les effets 
économiques négatifs comme l’hyperinflation provoquée par une nouvelle monnaie révolutionnaire.